# 준지도 학습 (semi-supervised) — 라벨 적을 때 미라벨 활용 (Colab)

**연습 기법** (IOAI 2025 Individual **Antique** 와 동일): 라벨이 *일부만* 있는 데이터에서 **미라벨 샘플까지 활용**해
분류한다(준지도 학습). Antique 는 라벨(진품 1/위작 -1) 외에 미상(0)이 많고, **미라벨의 구조(군집)** 를 써서 맞힌다.

**Antique 모범답안 = cluster-then-label**: `SpectralClustering` 로 *전체(라벨+미라벨)* 군집화 → 소수 라벨의 다수결로
각 군집에 라벨 부여(pseudo-label) → `SVC` 학습. 즉 **미라벨의 구조로 의사라벨을 만들어 학습**.

**대회**: [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer) (MNIST). 라벨을 **소수만** 남기고 나머지를
미라벨로 둔 뒤, 준지도 학습이 **소수 라벨만 쓴 지도학습을 이기는지** 본다.

| Antique 모범답안 (cluster-then-label) | 이 노트북 |
|---|---|
| 미라벨 구조로 **의사라벨 → 분류기 학습** | 동일 계열: **self-training**(의사라벨)·**label propagation** |
| `SpectralClustering`+다수결+`SVC` | MNIST 는 클래스가 *겹쳐* 순수 군집화가 약함 → 같은 *의사라벨/전파* 계열로 |

> ⚠️ **정직한 한계**: Antique 데이터는 *깨끗하게 군집되도록 설계*돼 cluster-then-label 이 통하지만, MNIST 숫자는
> 클래스가 겹쳐 *순수 군집→라벨* 은 지도학습보다 못하다(실측: 군집-라벨 0.5~0.76 < 지도 0.85). 그래서 핵심(미라벨
> 활용 SSL)은 유지하되, 실데이터에 강한 **같은 계열(self-training/label-propagation)** 로 시연한다.
> ⚙️ scikit-learn 만, T4 수 분. ⚠️ API 토큰 평문 — 외부 공유 금지.

## 0. 설치 · 자격증명 · 시드

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
import numpy as np, random
random.seed(42); np.random.seed(42)
print("준비 완료")

## 1. Kaggle 다운로드 & 특징(PCA) · 준지도 세팅

In [ ]:
import glob, zipfile, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
train = pd.read_csv(os.path.join(WORK_DIR, "train.csv")); test = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
y = train["label"].to_numpy()
X = train.drop(columns=["label"]).to_numpy(dtype=np.float32) / 255.
X_test = test.to_numpy(dtype=np.float32) / 255.

# PCA 특징 (train 으로 fit → train·test 변환)
scaler = StandardScaler().fit(X); pca = PCA(50, random_state=0).fit(scaler.transform(X))
Xp = pca.transform(scaler.transform(X)); Xtp = pca.transform(scaler.transform(X_test))

# 준지도 세팅: 평가용 val 분리 → 나머지에서 소수 NLAB 만 라벨, 나머지는 미라벨(-1)
pool, val = train_test_split(np.arange(len(X)), test_size=0.25, random_state=42, stratify=y)
NLAB = 100
lab = np.random.RandomState(0).choice(pool, NLAB, replace=False)
yl = np.full(len(X), -1); yl[lab] = y[lab]              # -1 = 미라벨
print(f"라벨 {NLAB}개 + 미라벨 {len(pool)-NLAB}개 | 평가 val {len(val)}개")

## 2. 기준선 — 소수 라벨만 쓴 지도학습

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
sup = SVC().fit(Xp[lab], y[lab])
acc_sup = accuracy_score(y[val], sup.predict(Xp[val]))
print(f"지도학습(라벨 {NLAB}개만)  val accuracy = {acc_sup:.4f}")

## 3. 준지도 — self-training(의사라벨) · label propagation (미라벨 활용)
**self-training**: 라벨로 분류기 학습 → 확신 높은 미라벨에 *의사라벨* 부여 → 다시 학습(Antique 의 의사라벨 계열).
**LabelSpreading**: kNN 그래프(데이터 구조)를 따라 라벨을 *전파*(Antique 의 '구조 활용'과 동형).

In [ ]:
from sklearn.semi_supervised import SelfTrainingClassifier, LabelSpreading
from sklearn.linear_model import LogisticRegression

# 학습에는 (소수 라벨 + 미라벨 pool) 사용. 평가는 별도 val.
fit_mask = np.zeros(len(X), bool); fit_mask[pool] = True

self_tr = SelfTrainingClassifier(LogisticRegression(max_iter=300))
self_tr.fit(Xp[fit_mask], yl[fit_mask])
acc_st = accuracy_score(y[val], self_tr.predict(Xp[val]))

ls = LabelSpreading(kernel="knn", n_neighbors=10)
ls.fit(Xp[fit_mask], yl[fit_mask])
acc_ls = accuracy_score(y[val], ls.predict(Xp[val]))

print(f"지도학습(few)        val accuracy = {acc_sup:.4f}")
print(f"self-training        val accuracy = {acc_st:.4f}   (Δ {acc_st-acc_sup:+.4f})")
print(f"label propagation    val accuracy = {acc_ls:.4f}   (Δ {acc_ls-acc_sup:+.4f})")
print("\n→ 미라벨을 활용해 소수 라벨 지도학습을 능가 — Antique 의 준지도(미라벨 활용) 핵심.")

## 4. (참고) Antique 모범답안 방식 — cluster-then-label, 그리고 한계
`KMeans` 로 전체 군집화 → 소수 라벨 다수결로 군집에 라벨 → 분류기 학습(모범답안과 동일 절차). MNIST 는 클래스가
겹쳐 군집이 깨끗하지 않아 *지도(few)보다 낮을 수 있다* — Antique 의 깨끗한 군집 데이터와 대비되는 정직한 한계.

In [ ]:
from sklearn.cluster import KMeans
from collections import Counter
K = 50                                                  # MNIST 는 겹쳐서 over-clustering
cl = KMeans(K, n_init=5, random_state=42).fit_predict(Xp[fit_mask])
yl_pool = yl[fit_mask]
c2l = {c: (Counter(yl_pool[(cl==c) & (yl_pool>=0)]).most_common(1)[0][0]
           if ((cl==c) & (yl_pool>=0)).any() else 0) for c in range(K)}
pseudo = np.array([c2l[c] for c in cl])
clf = SVC().fit(Xp[fit_mask], pseudo)
acc_cl = accuracy_score(y[val], clf.predict(Xp[val]))
print(f"cluster-then-label(모범답안 방식, K={K}) val accuracy = {acc_cl:.4f}")
print("→ MNIST 겹치는 클래스라 군집-라벨은 약함. Antique 깨끗한 군집 데이터에선 이 방식이 최적(0.98).")

## 5. 캐글 제출 — self-training 으로 test 예측 (`ImageId,Label`)
self-training 은 *귀납적*(새 데이터 예측 가능)이라 test 에 바로 적용. (소수 라벨 준지도 세팅이라 점수는 완전지도보다 낮음)

In [ ]:
test_pred = self_tr.predict(Xtp)
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"ImageId": np.arange(1, len(test_pred)+1), "Label": test_pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(test_pred))
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) 에 제출.

Antique 핵심(**준지도: 미라벨 활용**)을 digit-recognizer 로 연습 — 소수 라벨 지도학습 대비 **self-training/label
propagation 이 정확도↑**. 모범답안의 *cluster-then-label*(SpectralClustering+다수결+SVC)은 Antique 의 *깨끗한 군집*
데이터에 최적(0.98)이지만, MNIST 처럼 클래스가 겹치면 *의사라벨/전파* 계열이 더 낫다(같은 SSL 가족). 더: 라벨 수·
이웃 수·confidence 임계값 조정, 더 좋은 특징(CNN 임베딩).